In [2]:
%pip install pydantic
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from datetime import datetime


  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 1.8 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 2.0 MB/s eta 0:00:01
   ------------------------------------- -- 1.8/2.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 1.8 MB/s eta 0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)

   ---------- ----------------------------- 1/4 [pydantic-core]
   ---------- ----------------------------- 1/4 [pydantic-core]
   -------------------- ------------------- 2/4 [annotated-types]
   ------------------------------ --------- 3/4 [pydantic]
   ------------------------------ --------- 3/4 [pydantic]
   ------------------------------ --------- 

In [5]:
class User(BaseModel):
    id: int
    name: str
    email: str
    is_active: bool = True
    created_at: Optional[datetime] = None

valid_user = User(
    id=1,
    name="John Doe",
    email="john@example.in",
    created_at=datetime.now()
)

print(valid_user)
print(f"User ID: {valid_user.id}")
print(f"User Name: {valid_user.name}")

id=1 name='John Doe' email='john@example.in' is_active=True created_at=datetime.datetime(2025, 5, 21, 18, 30, 19, 159237)
User ID: 1
User Name: John Doe


In [6]:
try:
    invalid_user = User(
        id="not-an-integer",
        name="John Doe",
        email="john@example.in",
        created_at=datetime.now()
    )
except ValueError as e:
    print(f"Error: {e}")

Error: 1 validation error for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-an-integer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing


In [7]:
user_dict = valid_user.model_dump()
print(f"User as dictionary: {user_dict}")
user_json = valid_user.model_dump_json()
print(f"User as JSON: {user_json}")

User as dictionary: {'id': 1, 'name': 'John Doe', 'email': 'john@example.in', 'is_active': True, 'created_at': datetime.datetime(2025, 5, 21, 18, 30, 19, 159237)}
User as JSON: {"id":1,"name":"John Doe","email":"john@example.in","is_active":true,"created_at":"2025-05-21T18:30:19.159237"}


In [8]:
from pydantic import BaseModel, Field, validator, EmailStr
from typing import List, Optional
from datetime import datetime

class Product(BaseModel):
    # Field with constraints using Field
    id: int = Field(..., gt=0)  # '...' means required, gt=0 means greater than 0
    name: str = Field(..., min_length=3, max_length=50)
    price: float = Field(..., ge=0.0)  # 'ge' means greater than or equal to
    tags: List[str] = Field(default_factory=list)  # Default value as empty list
    description: Optional[str] = None
    
    # Custom validator for product name
    @validator('name')
    def name_must_not_contain_numbers(cls, v):
        if any(char.isdigit() for char in v):
            raise ValueError('name must not contain numeric characters')
        return v.title()  # Convert to title case
    
    # Custom validator that depends on multiple values
    @validator('price')
    def premium_products_need_description(cls, v, values):
        if v > 100.0 and not values.get('description'):
            raise ValueError('premium products (price > 100) must have a description')
        return v

# Create a regular product
regular_product = Product(
    id=1,
    name="coffee mug",
    price=12.99,
    tags=["kitchen", "ceramic"]
)
print(f"Regular product: {regular_product}")

Regular product: id=1 name='Coffee Mug' price=12.99 tags=['kitchen', 'ceramic'] description=None


C:\Users\choud\AppData\Local\Temp\ipykernel_18780\769948219.py:14: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  @validator('name')
C:\Users\choud\AppData\Local\Temp\ipykernel_18780\769948219.py:21: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  @validator('price')


In [9]:
try:
    invalid_name = Product(
        id=4,
        name="Blender 3000",
        price=49.99
    )
except Exception as e:
    print(f"Validation error (name with numbers): {e}")

Validation error (name with numbers): 1 validation error for Product
name
  Value error, name must not contain numeric characters [type=value_error, input_value='Blender 3000', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error
